# Урок 1 — Домашнее задание 10

**Генерация данных и сортировка: пузырёк + `list.sort()` с lambda**



## 1. Постановка задачи

1. Сгенерировать массив из **100 000** случайных целых чисел
   (`random.randint`, диапазон `1..1_000_000`).
2. С помощью `range()` сгенерировать второй массив — **100 000 словарей**
   со структурой `{"num_1": <int>, "num_2": <int>}`.
3. Реализовать функцию **сортировки пузырьком** и отсортировать ей первый массив.
4. Второй массив отсортировать встроенным методом **`.sort()`** и
   **лямбда-функцией**: сначала по ключу `num_1`, затем по ключу `num_2`.

## 2. Конфигурация

В Colab аргументы командной строки не используются, поэтому объёмы задаются
константами. Пузырьковая сортировка на 100 000 элементах занимает
**несколько минут** — для быстрой проверки поставьте `BUBBLE_N = 5_000`.

In [1]:
import random
import sys
import time

SIZE = 100_000          # размер коллекций по заданию
MAX_VALUE = 1_000_000   # верхняя граница random.randint (включительно)
BUBBLE_N = 100_000      # сколько элементов сортировать пузырьком
RANDOM_SEED = None      # например 42, чтобы прогоны были воспроизводимы

print("Python:", sys.version.split()[0])
if BUBBLE_N >= 50_000:
    print("Внимание: пузырёк на %d элементах займет несколько минут." % BUBBLE_N)
    print("Для быстрой проверки поставьте BUBBLE_N = 5_000.")

Python: 3.13.15
Внимание: пузырёк на 100000 элементах займет несколько минут.
Для быстрой проверки поставьте BUBBLE_N = 5_000.


## 3. Генерация данных

`range(n)` выступает счётчиком повторений в списковых включениях, поэтому
оба массива содержат ровно `n` элементов. `random.randint(1, MAX_VALUE)`
включает обе границы.

In [2]:
def make_numbers(n):
    """Список из n случайных целых чисел от 1 до MAX_VALUE (через range)."""
    return [random.randint(1, MAX_VALUE) for _ in range(n)]


def make_dicts(n):
    """Список из n словарей {"num_1": ..., "num_2": ...}, тоже через range."""
    return [
        {
            "num_1": random.randint(1, MAX_VALUE),
            "num_2": random.randint(1, MAX_VALUE),
        }
        for _ in range(n)
    ]


random.seed(RANDOM_SEED)

start = time.perf_counter()
numbers = make_numbers(SIZE)
records = make_dicts(SIZE)
gen_time = time.perf_counter() - start

print("Длина первого массива :", len(numbers))
print("Длина второго массива :", len(records))
print("Диапазон значений     :", min(numbers), "-", max(numbers))
print("Первые числа          :", numbers[:5])
print("Первые словари        :", records[:2])
print("Время генерации       : %.3f c" % gen_time)

Длина первого массива : 100000
Длина второго массива : 100000
Диапазон значений     : 1 - 999992
Первые числа          : [940304, 338585, 41533, 364703, 501266]
Первые словари        : [{'num_1': 675480, 'num_2': 832029}, {'num_1': 240818, 'num_2': 625952}]
Время генерации       : 0.415 c


## 4. Сортировка пузырьком

Классический алгоритм — два вложенных цикла и сравнение соседних элементов.
Добавлены две оптимизации, не меняющие смысла:

* `border` сжимается до позиции **последнего обмена**: хвост после последней
  перестановки уже упорядочен, повторно по нему проходить не нужно;
* если за проход обменов не было, `border` становится нулём и сортировка
  завершается досрочно (лучший случай — `O(n)`).

Сложность: `O(n^2)` в среднем и худшем случае, памяти `O(1)` сверх копии.
Алгоритм устойчив. Функция возвращает новый список, исходный не мутируется.

In [3]:
def bubble_sort(items):
    """Пузырьковая сортировка: возвращает новый список, исходный не меняется."""
    result = list(items)
    border = len(result) - 1

    while border > 0:
        last_swap = 0

        for i in range(border):
            if result[i] > result[i + 1]:
                result[i], result[i + 1] = result[i + 1], result[i]
                last_swap = i

        border = last_swap

    return result


assert bubble_sort([]) == []
assert bubble_sort([5]) == [5]
assert bubble_sort([3, 3, 1, 0, -7, 2]) == [-7, 0, 1, 2, 3, 3]
source = [4, 2, 9, 1]
assert bubble_sort(source) == [1, 2, 4, 9]
assert source == [4, 2, 9, 1], "исходный список не должен меняться"
print("Тесты пузырьковой сортировки пройдены")


Тесты пузырьковой сортировки пройдены


## 5. Сортировка первого массива пузырьком

Результат сверяется со встроенным `sorted()` — независимая проверка
корректности на реальном объёме данных.

In [ ]:
sample = numbers[:BUBBLE_N]

start = time.perf_counter()
sorted_sample = bubble_sort(sample)
bubble_time = time.perf_counter() - start

start = time.perf_counter()
_ = sorted(sample)
builtin_time = time.perf_counter() - start

print("n                 : %d" % len(sample))
print("пузырёк           : %.3f c" % bubble_time)
print("встроенный sorted : %.4f c" % builtin_time)
print("результат верный  : %s" % (sorted_sample == sorted(sample)))
print("во сколько раз sorted быстрее: x%.0f" % (bubble_time / builtin_time))

## 6. Сортировка второго массива: `.sort()` + lambda

`list.sort()` использует Timsort — `O(n log n)` и является устойчивой сортировкой.
Для последовательной сортировки по двум ключам сначала сортируют по второстепенному ключу `num_2`, затем по основному `num_1`.
Составную сортировку по `(num_1, num_2)` можно задать одним вызовом с кортежем в качестве ключа.


In [ ]:
start = time.perf_counter()
records.sort(key=lambda item: item["num_1"])
t_num1 = time.perf_counter() - start
print("sort по num_1 : %.4f c" % t_num1)
print("начало списка :", records[:3])

start = time.perf_counter()
records.sort(key=lambda item: item["num_2"])
t_num2 = time.perf_counter() - start
print("sort по num_2 : %.4f c" % t_num2)
print("начало списка :", records[:3])

values2 = [item["num_2"] for item in records]
print("отсортирован по num_2       :", values2 == sorted(values2))

records.sort(key=lambda item: (item["num_1"], item["num_2"]))
pairs = [(item["num_1"], item["num_2"]) for item in records]
print("отсортирован по (num_1, num_2):", pairs == sorted(pairs))

## 7. Почему пузырёк непригоден для больших объёмов

Замерим пузырёк на растущих `n` и построим график в логарифмическом масштабе:
прямая с наклоном 2 подтверждает квадратичный рост `O(n^2)`. По калибровке
экстраполируем время для 100 000 элементов.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

sizes = [250, 500, 1000, 2000, 4000]
times = []
for n in sizes:
    data = random.sample(range(MAX_VALUE), n)
    start = time.perf_counter()
    bubble_sort(data)
    times.append(time.perf_counter() - start)
    print("n=%5d  время=%.3f c" % (n, times[-1]))

k = times[-1] / sizes[-1] ** 2
print("Оценка времени пузырька для 100 000 элементов: %.0f c" % (k * 100_000 ** 2))

if plt is not None:
    reference = [times[0] * (n / sizes[0]) ** 2 for n in sizes]
    plt.figure(figsize=(6, 4))
    plt.loglog(sizes, times, "o-", label="пузырёк (замер)")
    plt.loglog(sizes, reference, "--", label="эталон n^2")
    plt.xlabel("n")
    plt.ylabel("секунды")
    plt.grid(True, which="both")
    plt.legend()
    plt.title("Квадратичный рост времени пузырьковой сортировки")
    plt.show()
else:
    print("matplotlib недоступен - график пропущен")

### 7.1 Результаты прогона на локальной машине (для сверки)

`sorting_task.py`, Python 3.14.7, Windows x64, `n = 100 000`:

```
bubble sort: 319.345 s, correct = True
sort by num_1: 0.0193 s, head: [{'num_1': 17, ...}, {'num_1': 38, ...}, {'num_1': 49, ...}]
sort by num_2: 0.0382 s, head: [{'num_2': 3}, {'num_2': 6}, {'num_2': 34}]
```

Калибровка одной итерации пузырька: ~84 нс, то есть без сжатия границы для
100 000 элементов получилось бы около 840 c. Сжатие границы дало 319 c.
В Colab абсолютные значения будут другими (другой CPU), порядок величины тот же.

## 8. Выводы

* Пузырёк корректен, но на 100 000 элементов требует **минут** против
  **сотых долей секунды** у `list.sort()` — разница порядка десятков тысяч раз.
* Устойчивость Timsort позволяет строить составную сортировку последовательными
  вызовами `.sort()` в обратном порядке приоритета ключей.
* В реальных задачах пузырьковую сортировку используют только как учебный
  пример либо при заведомо малом объёме данных.

In [ ]:
report = "\n".join([
    "Урок 1 - ДЗ 10: результаты прогона",
    "Python: " + sys.version.split()[0],
    "SIZE = %d, BUBBLE_N = %d" % (SIZE, BUBBLE_N),
    "пузырёк            : %.3f c" % bubble_time,
    "встроенный sorted  : %.4f c" % builtin_time,
    "sort по num_1      : %.4f c" % t_num1,
    "sort по num_2      : %.4f c" % t_num2,
])
print(report)

with open("lesson_1_ДЗ_10_results.txt", "w", encoding="utf-8") as f:
    f.write(report + "\n")
print("Сохранено в рабочую папку окружения: lesson_1_ДЗ_10_results.txt")

# Для скачивания результата в Colab раскомментируйте:
# from google.colab import files
# files.download("lesson_1_ДЗ_10_results.txt")